This is Tested model

# Job Recommendation System

A two-phase TF-IDF–based system:
- **Phase 1 – Job-to-Job similarity**: find jobs similar to a given posting.
- **Phase 2 – Resume-to-Job matching**: rank candidates against every job.

## 1. Imports & NLTK setup

In [1]:
import re
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("All imports successful.")

All imports successful.


## 2. Text preprocessing

In [2]:
_stop_words = set(stopwords.words('english'))
_lemmatizer = WordNetLemmatizer()


def preprocess(text: str) -> str:
    """Clean and normalise a text string in a single pass.

    Steps:
    1. Lower-case
    2. Strip non-alpha characters
    3. Collapse whitespace
    4. Remove English stop-words
    5. Lemmatise each token

    Returns an empty string for missing / non-string input.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # keep only letters + spaces
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = [
        _lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in _stop_words and len(w) > 1   # drop single-char noise
    ]
    return ' '.join(tokens)

## 3. Load & prepare job data

In [3]:
jobs_df = pd.read_csv("clean_job_dataset.csv")

# Build processed_text from Job Description if not already present
if 'processed_text' not in jobs_df.columns:
    jobs_df['processed_text'] = jobs_df['Job Description'].apply(preprocess)

# Drop rows with empty processed text and reset index
jobs_df = (
    jobs_df[jobs_df['processed_text'].str.strip().astype(bool)]
    .reset_index(drop=True)
)

print(f"Loaded {len(jobs_df):,} job postings.")
jobs_df.head()

Loaded 1,615,940 job postings.


,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile,cleaned_text,processed_text
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...",NaN,"Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie...",social media managers oversee an organizations...,social medium manager oversee organization soc...
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...",NaN,"Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com...",frontend web developers design and implement u...,frontend web developer design implement user i...
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",NaN,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P...",quality control managers establish and enforce...,quality control manager establish enforce qual...
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",communication,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O...",wireless network engineers design implement an...,wireless network engineer design implement mai...
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",NaN,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ...",a conference manager coordinates and manages c...,conference manager coordinate manage conferenc...


## 4. Phase 1 – Baseline: Job-to-Job similarity

Vectorise all job descriptions with TF-IDF (unigrams + bigrams, 5 000 features).  
**We do NOT pre-compute the full N×N similarity matrix** — with ~1 M jobs that  
would require ~9 TiB of RAM. Instead, similarity is computed on-demand per query  
by dotting a single job vector against the entire sparse matrix.

In [4]:
vectorizer_p1 = TfidfVectorizer(
    max_features=5_000,     # was 100 — far too low to capture vocabulary
    ngram_range=(1, 2),     # unigrams + bigrams for richer matching
    sublinear_tf=True,      # apply log(1 + tf) to dampen high-frequency terms
    stop_words='english',
)

# Keep the sparse matrix — DO NOT call linear_kernel/cosine_similarity on the
# full matrix here.  With ~1 M rows that produces an N×N dense array (~9 TiB).
job_tfidf = vectorizer_p1.fit_transform(jobs_df['processed_text'])

print(f"TF-IDF matrix: {job_tfidf.shape}  "
      f"({job_tfidf.nnz:,} non-zero elements, "
      f"{job_tfidf.data.nbytes / 1e6:.1f} MB in memory)")

TF-IDF matrix: (1615940, 5000)  (48,688,527 non-zero elements, 389.5 MB in memory)


In [5]:
def recommend_jobs(job_index: int, top_n: int = 5) -> pd.DataFrame:
    """Return the top-N most similar jobs to the job at `job_index`.

    Computes similarity on-demand for a single query vector — no full
    N×N matrix is ever built, so this works with millions of job rows.

    Parameters
    ----------
    job_index : int
        Row index in jobs_df.
    top_n : int
        Number of recommendations to return (default 5).

    Returns
    -------
    pd.DataFrame with columns [Job Title, Company, similarity_score].
    """
    # Extract the single query vector (1 × vocab) — stays sparse
    query_vec = job_tfidf[job_index]

    # Dot product against all job vectors → cosine scores (1-D array)
    # linear_kernel on a (1, V) × (N, V).T is O(N·V), not O(N²)
    scores = linear_kernel(query_vec, job_tfidf).flatten()

    # Use argpartition to find top-(top_n+1) without full sort — O(N) not O(N log N)
    top_n1 = top_n + 1
    part = np.argpartition(-scores, top_n1)[:top_n1]
    top_idx = part[np.argsort(-scores[part])]   # sort only the small partition
    top_idx = top_idx[top_idx != job_index][:top_n]   # exclude self

    result = jobs_df.iloc[top_idx][['Job Title', 'Company']].copy()
    result['similarity_score'] = scores[top_idx].round(4)
    result.index = range(1, len(result) + 1)   # 1-based ranking
    return result

In [6]:
# --- Test Phase 1 ---
seed_job = 0
print(f"Seed job: '{jobs_df.iloc[seed_job]['Job Title']}' @ {jobs_df.iloc[seed_job]['Company']}\n")
recommend_jobs(seed_job, top_n=5)

Seed job: 'Digital Marketing Specialist' @ Icahn Enterprises



,Job Title,Company,similarity_score
1,Digital Marketing Specialist,DISH Network,1.0
2,Digital Marketing Specialist,Goodyear Tire & Rubber,1.0
3,Digital Marketing Specialist,Agricultural Bank of China,1.0
4,Digital Marketing Specialist,Salesforce,1.0
5,Digital Marketing Specialist,American Electric Power,1.0


## 5. Phase 2 – Advanced: Resume-to-Job matching

Fit a shared TF-IDF vocabulary on **all** text (resumes + jobs), then  
compute a cross-similarity matrix **(resumes × jobs)**.  
Results are built with vectorised NumPy ops instead of a Python loop.

In [7]:
resume_data = [
    {"name": "Candidate 1", "resume_text": "Python developer with machine learning and NLP experience"},
    {"name": "Candidate 2", "resume_text": "Java backend developer with Spring Boot and SQL"},
    {"name": "Candidate 3", "resume_text": "Data analyst with Excel, SQL and Power BI"},
    {"name": "Candidate 4", "resume_text": "AI engineer with deep learning and PyTorch"},
    {"name": "Candidate 5", "resume_text": "Frontend developer with React and JavaScript"},
]

resume_df = pd.DataFrame(resume_data)
resume_df['processed_text'] = resume_df['resume_text'].apply(preprocess)
resume_df[['name', 'processed_text']]

,name,processed_text
0,Candidate 1,python developer machine learning nlp experience
1,Candidate 2,java backend developer spring boot sql
2,Candidate 3,data analyst excel sql power bi
3,Candidate 4,ai engineer deep learning pytorch
4,Candidate 5,frontend developer react javascript


In [8]:
vectorizer_p2 = TfidfVectorizer(
    max_features=5_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english',
)

# Fit on the union of resumes + jobs so both share the same feature space
combined_text = resume_df['processed_text'].tolist() + jobs_df['processed_text'].tolist()
tfidf_matrix = vectorizer_p2.fit_transform(combined_text)

n_resumes = len(resume_df)
resume_vectors = tfidf_matrix[:n_resumes]          # shape: (R, vocab)
job_vectors    = tfidf_matrix[n_resumes:]           # shape: (J, vocab)

# Cross-similarity: (R, J)  — each row = one resume, each col = one job
resume_job_sim = cosine_similarity(resume_vectors, job_vectors)

print(f"Resume-Job similarity matrix: {resume_job_sim.shape}  (resumes × jobs)")

Resume-Job similarity matrix: (5, 1615940)  (resumes × jobs)


In [9]:
# Vectorised ranking: for every job find the top-3 candidates
TOP_K = 3

# Argsort descending along the resume axis (axis=0)
top_candidate_idx = np.argsort(-resume_job_sim, axis=0)[:TOP_K]   # (TOP_K, J)

records = []
for j in range(len(jobs_df)):
    for rank, c_idx in enumerate(top_candidate_idx[:, j], start=1):
        records.append({
            'job_title':  jobs_df.iloc[j]['Job Title'],
            'company':    jobs_df.iloc[j]['Company'],
            'rank':       rank,
            'candidate':  resume_df.iloc[c_idx]['name'],
            'score':      round(resume_job_sim[c_idx, j], 4),
        })

results_df = pd.DataFrame(records)
print(f"Total match records: {len(results_df):,}")
results_df.head(15)

Total match records: 4,847,820


,job_title,company,rank,candidate,score
0,Digital Marketing Specialist,Icahn Enterprises,1,Candidate 1,0.0000
1,Digital Marketing Specialist,Icahn Enterprises,2,Candidate 2,0.0000
2,Digital Marketing Specialist,Icahn Enterprises,3,Candidate 3,0.0000
3,Web Developer,PNC Financial Services Group,1,Candidate 2,0.1656
4,Web Developer,PNC Financial Services Group,2,Candidate 5,0.1405
5,Web Developer,PNC Financial Services Group,3,Candidate 1,0.0938
6,Operations Manager,United Services Automobile Assn.,1,Candidate 1,0.0000
7,Operations Manager,United Services Automobile Assn.,2,Candidate 2,0.0000
8,Operations Manager,United Services Automobile Assn.,3,Candidate 3,0.0000
9,Network Engineer,Hess,1,Candidate 4,0.0361


In [10]:
# Best (rank-1) candidate per job, sorted by match score descending
best_candidates = (
    results_df[results_df['rank'] == 1]
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

best_candidates.head(10)

,job_title,company,rank,candidate,score
0,Java Developer,Royal Caribbean Group,1,Candidate 2,0.3547
1,Java Developer,Constellation Energy,1,Candidate 2,0.3547
2,Java Developer,Williams,1,Candidate 2,0.3547
3,Java Developer,Woodside Petroleum,1,Candidate 2,0.3547
4,Java Developer,Altria Group,1,Candidate 2,0.3547
5,Java Developer,Elevance Health,1,Candidate 2,0.3547
6,Java Developer,Sky plc,1,Candidate 2,0.3547
7,Java Developer,Apple,1,Candidate 2,0.3547
8,Java Developer,Warner Bros. Discovery,1,Candidate 2,0.3547
9,Java Developer,China Shenhua Energy,1,Candidate 2,0.3547


## 6. Top jobs per candidate

In [11]:
def top_jobs_for_candidate(candidate_name: str, top_n: int = 5) -> pd.DataFrame:
    """Return the best-matching jobs for a given candidate."""
    c_idx = resume_df.index[resume_df['name'] == candidate_name].item()
    scores = resume_job_sim[c_idx]                  # shape: (J,)
    top_idx = np.argsort(-scores)[:top_n]

    out = jobs_df.iloc[top_idx][['Job Title', 'Company']].copy()
    out['match_score'] = scores[top_idx].round(4)
    out.index = range(1, len(out) + 1)
    return out


for candidate in resume_df['name']:
    print(f"\n=== Top 3 jobs for {candidate} ===")
    print(top_jobs_for_candidate(candidate, top_n=3).to_string())


=== Top 3 jobs for Candidate 1 ===
        Job Title           Company  match_score
1  Data Scientist  Deutsche Bahn AG       0.3416
2  Data Scientist   Cheniere Energy       0.3416
3  Data Scientist        Mastercard       0.3416

=== Top 3 jobs for Candidate 2 ===
        Job Title                      Company  match_score
1  Java Developer  Discover Financial Services       0.3547
2  Java Developer                          KLA       0.3547
3  Java Developer         Edison International       0.3547

=== Top 3 jobs for Candidate 3 ===
             Job Title                Company  match_score
1  Electrical Engineer          National Grid       0.1898
2  Electrical Engineer  Harley-Davidson, Inc.       0.1898
3  Electrical Engineer             Siemens AG       0.1898

=== Top 3 jobs for Candidate 4 ===
      Job Title           Company  match_score
1  Psychologist   Skechers U.S.A.       0.1533
2  Psychologist       TPG Telecom       0.1533
3  Psychologist  UltraTech Cement       0.1